# Geoencoding Dataset Municipalities

In [1]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from rapidfuzz import process, fuzz
import unidecode
import re

### Dataset Municipalities

In [2]:
categories = pd.read_parquet(r'../export/small_parquet/municiplity.parquet')
categories

,DE_MUNICIP
0,ABRERA
1,ACEBEDO
2,ADEJE CASCO
3,AGUILON
4,ALAMEDA DE LA SAGRA
...,...
804,VILLASILA DE VALDAVIA
805,VILLAZANZO DE VALDERADUEY
806,VITORIA-GASTEIZ
807,VIVEIRO


In [3]:
categories.nunique()

DE_MUNICIP    809
dtype: int64

In [4]:
categories.dtypes

DE_MUNICIP    object
dtype: object

In [5]:
categories['DE_MUNICIP'] = categories['DE_MUNICIP'].str.strip()

In [6]:
categories.sort_values(by='DE_MUNICIP', inplace=True)
categories.head()

,DE_MUNICIP
201,
202,A CORUÃ‘A
203,ABIA DE LAS TORRES
0,ABRERA
1,ACEBEDO


In [7]:
categories.isna().sum()

DE_MUNICIP    0
dtype: int64

In [8]:
categories = categories[categories['DE_MUNICIP'].str.strip() != '']
categories.shape

(808, 1)

### National Geographic InstituteAutonomous Body National Center for Geographic Information
https://centrodedescargas.cnig.es/CentroDescargas/nomenclator-geografico-municipios-entidades-poblacion

In [9]:
latlong_data = pd.read_csv(r"..\BD_MUNICIPIOS-ENTIDADES\MUNICIPIOS.csv",
                           delimiter=';',
                           encoding='latin1')
latlong_data.head()

,COD_INE,ID_REL,COD_GEO,COD_PROV,PROVINCIA,NOMBRE_ACTUAL,POBLACION_MUNI,SUPERFICIE,PERIMETRO,COD_INE_CAPITAL,CAPITAL,POBLACION_CAPITAL,HOJA_MTN25,LONGITUD_ETRS89_REGCAN95,LATITUD_ETRS89_REGCAN95,ORIGENCOOR,ALTITUD,ORIGENALTITUD
0,1001000000,1010014,1010,1,Araba/Álava,Alegría-Dulantzi,2961,"1994,5872",35069,1001000101,Alegría-Dulantzi,2842,0113-3,"-2,512507724","42,84045247",Detección automática,568,MDT
1,1002000000,1010029,1020,1,Araba/Álava,Amurrio,10346,"9617,86",65701,1002000201,Amurrio,9256,0086-4,"-3,001015194","43,05265767",Detección automática,217,MDT
2,1003000000,1010035,1030,1,Araba/Álava,Aramaio,1353,"7308,96",42097,1003000601,Ibarra,731,0087-4,"-2,564829379","43,05257873",Detección automática,325,MDT
3,1004000000,1010040,1040,1,Araba/Álava,Artziniega,1868,"2728,73",22886,1004000101,Artziniega,1732,0086-1,"-3,13052099","43,1217919",Detección automática,199,MDT
4,1006000000,1010066,1060,1,Araba/Álava,Armiñón,233,"1297,27",24707,1006000101,Armiñón,106,0137-4,"-2,872270813","42,72340924",Detección automática,466,MDT


In [10]:
latlong_data = latlong_data[['COD_INE', 'COD_PROV','NOMBRE_ACTUAL', 'PROVINCIA', 'LONGITUD_ETRS89_REGCAN95', 'LATITUD_ETRS89_REGCAN95', 'ALTITUD']]
latlong_data.shape

(8132, 7)

In [11]:
latlong_data.rename(columns={
    'COD_INE':'INE_CODE',
    'COD_PROV':'PROVINCE_CODE',
    'NOMBRE_ACTUAL':'DE_MUNICIP',
    'PROVINCIA':'PROVINCE',
    'LONGITUD_ETRS89_REGCAN95':'LONGITUDE',
    'LATITUD_ETRS89_REGCAN95':'LATITUDE',
    'ALTITUD':'ALTITUDE',
},
inplace=True)
latlong_data.head()

,INE_CODE,PROVINCE_CODE,DE_MUNICIP,PROVINCE,LONGITUDE,LATITUDE,ALTITUDE
0,1001000000,1,Alegría-Dulantzi,Araba/Álava,"-2,512507724","42,84045247",568
1,1002000000,1,Amurrio,Araba/Álava,"-3,001015194","43,05265767",217
2,1003000000,1,Aramaio,Araba/Álava,"-2,564829379","43,05257873",325
3,1004000000,1,Artziniega,Araba/Álava,"-3,13052099","43,1217919",199
4,1006000000,1,Armiñón,Araba/Álava,"-2,872270813","42,72340924",466


In [12]:
latlong_data.dtypes

INE_CODE          int64
PROVINCE_CODE     int64
DE_MUNICIP       object
PROVINCE         object
LONGITUDE        object
LATITUDE         object
ALTITUDE         object
dtype: object

In [13]:
latlong_data['DE_MUNICIP'] = latlong_data['DE_MUNICIP'].str.strip().str.upper()
latlong_data['PROVINCE'] = latlong_data['PROVINCE'].str.strip()
latlong_data['LATITUDE'] = latlong_data['LATITUDE'].str.replace(',', '.').astype('float64')
latlong_data['LONGITUDE'] = latlong_data['LONGITUDE'].str.replace(',', '.').astype('float64')
print(latlong_data.dtypes)
latlong_data.head()

INE_CODE           int64
PROVINCE_CODE      int64
DE_MUNICIP        object
PROVINCE          object
LONGITUDE        float64
LATITUDE         float64
ALTITUDE          object
dtype: object


,INE_CODE,PROVINCE_CODE,DE_MUNICIP,PROVINCE,LONGITUDE,LATITUDE,ALTITUDE
0,1001000000,1,ALEGRÍA-DULANTZI,Araba/Álava,-2.512508,42.840452,568
1,1002000000,1,AMURRIO,Araba/Álava,-3.001015,43.052658,217
2,1003000000,1,ARAMAIO,Araba/Álava,-2.564829,43.052579,325
3,1004000000,1,ARTZINIEGA,Araba/Álava,-3.130521,43.121792,199
4,1006000000,1,ARMIÑÓN,Araba/Álava,-2.872271,42.723409,466


In [14]:
latlong_data.sort_values(by='DE_MUNICIP', inplace=True)
latlong_data.head()

,INE_CODE,PROVINCE_CODE,DE_MUNICIP,PROVINCE,LONGITUDE,LATITUDE,ALTITUDE
4891,32003000000,32,A ARNOIA,Ourense,-8.135751,42.253066,101
2132,15007000000,15,A BAÑA,A Coruña,-8.758003,42.961898,272
4902,32014000000,32,A BOLA,Ourense,-7.928518,42.140937,479
2143,15018000000,15,A CAPELA,A Coruña,-8.068806,43.435552,389
5292,36009000000,36,A CAÑIZA,Pontevedra,-8.273359,42.212754,568


In [15]:
merged = categories.merge(latlong_data, on=['DE_MUNICIP'], how='left')
merged

,DE_MUNICIP,INE_CODE,PROVINCE_CODE,PROVINCE,LONGITUDE,LATITUDE,ALTITUDE
0,A CORUÃ‘A,NaN,NaN,NaN,NaN,NaN,NaN
1,ABIA DE LAS TORRES,3.400300e+10,34.0,Palencia,-4.421902,42.420220,836
2,ABRERA,8.001000e+09,8.0,Barcelona,1.901569,41.516398,106
3,ACEBEDO,2.400100e+10,24.0,León,-5.115871,43.040732,1148
4,ACEUCHAL,6.002000e+09,6.0,Badajoz,-6.487251,38.647755,303
...,...,...,...,...,...,...,...
809,YUNQUERA,2.910000e+10,29.0,Málaga,-4.917332,36.734203,695
810,ZAFRA,6.158000e+09,6.0,Badajoz,-6.418559,38.426344,509
811,ZARAGOZA,5.029700e+10,50.0,Zaragoza,-0.877318,41.656208,208
812,ZARZOSA DE RIO PISUERGA,NaN,NaN,NaN,NaN,NaN,NaN


In [16]:
merged[merged['LONGITUDE'].isna()]

,DE_MUNICIP,INE_CODE,PROVINCE_CODE,PROVINCE,LONGITUDE,LATITUDE,ALTITUDE
0,A CORUÃ‘A,NaN,NaN,NaN,NaN,NaN,NaN
6,ADEJE CASCO,NaN,NaN,NaN,NaN,NaN,NaN
10,AGUILON,NaN,NaN,NaN,NaN,NaN,NaN
11,AGUIMES,NaN,NaN,NaN,NaN,NaN,NaN
12,AGÃœIMES,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...
798,VILLAYUSO,NaN,NaN,NaN,NaN,NaN,NaN
800,VILOBI D'ONYAR,NaN,NaN,NaN,NaN,NaN,NaN
801,VINAROS,NaN,NaN,NaN,NaN,NaN,NaN
805,XUNQUEIRA DE AMBIA,NaN,NaN,NaN,NaN,NaN,NaN


In [17]:
merged[merged['DE_MUNICIP'].duplicated(keep=False)]

,DE_MUNICIP,INE_CODE,PROVINCE_CODE,PROVINCE,LONGITUDE,LATITUDE,ALTITUDE
131,CABANES,1.203300e+10,12.0,Castelló/Castellón,0.045412,40.156104,290
132,CABANES,1.703000e+10,17.0,Girona,2.977957,42.307504,24
199,CIEZA,3.001900e+10,30.0,Murcia,-1.427730,38.236594,187
200,CIEZA,3.902100e+10,39.0,Cantabria,-4.096722,43.221056,174
433,MIERES,1.710500e+10,17.0,Girona,2.640292,42.123296,282
434,MIERES,3.303700e+10,33.0,Asturias,-5.772657,43.248794,210
579,SADA,3.121200e+10,31.0,Navarra,-1.397668,42.585825,479
580,SADA,1.507500e+10,15.0,A Coruña,-8.253590,43.351531,6
718,TORRENT,4.624400e+10,46.0,València/Valencia,-0.465982,39.436721,46
719,TORRENT,1.719700e+10,17.0,Girona,3.128197,41.951843,38


### Dataset Municipality-Province data

In [18]:
categories_prov = pd.read_parquet(r'..\export\small_parquet\muncipility_province.parquet')
print(categories_prov.shape)
categories_prov.head()

(778, 2)


,Municipality,Province
0,Abla,Almería
1,Abrucena,Almería
2,Adamuz,Córdoba
3,Adra,Almería
4,Agrón,Granada


In [19]:
same_name_diff_prov = merged[merged['DE_MUNICIP'].duplicated()]['DE_MUNICIP'].values
same_name_diff_prov

array(['CABANES', 'CIEZA', 'MIERES', 'SADA', 'TORRENT', 'VILLAESCUSA'],
      dtype=object)

In [20]:
categories_prov[categories_prov['Municipality'].isin(same_name_diff_prov)]

,Municipality,Province


### Fuzzy string similarity to merge on DE_MUNICIP names

In [21]:
def clean_municipality(name):
    if not isinstance(name, str): return ""
    # 1. Fix specific encoding artifacts
    name = name.replace("Ã‘", "Ñ").replace("Ã“", "Ó")
    
    # 2. Expand common abbreviations
    name = name.replace("S/C", "SANTA CRUZ")
    name = name.replace("S.", "SAN")
    
    # 3. Fix trailing articles e.g., "CUERVO (EL)" -> "EL CUERVO"
    name = re.sub(r'^(.*?)\s*\((EL|LA|LOS|LAS)\)$', r'\2 \1', name)
    
    # 4. Remove accents and convert to uppercase for baseline comparison
    name = unidecode.unidecode(name).upper()
    return name.strip()

# Apply cleaning to both lists
clean_dataset = [clean_municipality(name) for name in categories.DE_MUNICIP.tolist()]
clean_all = [clean_municipality(name) for name in latlong_data['DE_MUNICIP'].tolist()]

matches = {}
for original_name, clean_name in zip(categories.DE_MUNICIP.tolist(), clean_dataset):
    
    # Handle bilingual names by splitting at '/'
    parts = clean_name.split('/')
    best_match = None
    best_score = 0
    
    for part in parts:
        result = process.extractOne(
            part,
            clean_all,
            scorer=fuzz.token_sort_ratio,
            score_cutoff=85  # Raised significantly to prevent false positives
        )
        if result and result[1] > best_score:
            best_match = result
            best_score = result[1]
            
    if best_match:
        # Retrieve the original uncleaned name from the target dataframe using the index
        target_original_name = latlong_data['DE_MUNICIP'].iloc[best_match[2]]
        matches[original_name] = target_original_name
    else:
        matches[original_name] = None

In [22]:
matches

{'A CORUÃ‘A': 'A CORUÑA',
 'ABIA DE LAS TORRES': 'ABIA DE LAS TORRES',
 'ABRERA': 'ABRERA',
 'ACEBEDO': 'ACEBEDO',
 'ACEUCHAL': 'ACEUCHAL',
 'ADEJE': 'ADEJE',
 'ADEJE CASCO': None,
 'ADEMUZ': 'ADEMUZ',
 'AGRAMUNT': 'AGRAMUNT',
 'AGUADULCE': 'AGUADULCE',
 'AGUILON': 'AGUILÓN',
 'AGUIMES': 'AGÜIMES',
 'AGÃœIMES': None,
 'AITONA': 'AITONA',
 'AJO': None,
 'ALAMEDA DE LA SAGRA': 'ALAMEDA DE LA SAGRA',
 'ALARO': 'ALARÓ',
 'ALBACETE': 'ALBACETE',
 'ALBENDIEGO': 'ALBENDIEGO',
 'ALBOCASSER': 'ALBOCÀSSER',
 'ALBOLOTE': 'ALBOLOTE',
 'ALBURQUERQUE': 'ALBURQUERQUE',
 'ALCALA DE HENARES': 'ALCALÁ DE HENARES',
 'ALCALA DE XIVERT': 'ALCALÀ DE XIVERT',
 'ALCALA LA REAL': 'ALCALÁ LA REAL',
 'ALCANAR': 'ALCANAR',
 'ALCAUDETE': 'ALCAUDETE',
 'ALCAZAR DE SAN JUAN': 'ALCÁZAR DE SAN JUAN',
 'ALCAÃ‘IZ': 'ALCAÑIZ',
 'ALCOBENDAS': 'ALCOBENDAS',
 'ALCORCON': 'ALCORCÓN',
 'ALCUBIERRE': 'ALCUBIERRE',
 'ALCUDIA': 'ALCÚDIA',
 'ALDEASECA DE ALBA': 'ALDEASECA DE ALBA',
 'ALDEATEJADA': 'ALDEATEJADA',
 'ALFORJA': 'ALFO

In [23]:
manual_corrections = {
    'ADEJE CASCO': 'ADEJE',
    'AGÃœIMES': 'AGÜIMES',
    'AJO': 'BAREYO', 
    'ALICANTE/ALACANT': 'ALACANT/ALICANTE',
    'ALMAZORA': 'ALMASSORA',
    'ALQUERIAS DEL NIÃ‘O PERDIDO': 'LES ALQUERIES/ALQUERÍAS DEL NIÑO PERDIDO',
    'BARRIO DE BRICIA': 'ALFOZ DE BRICIA',
    'BENICASIM': 'BENICÀSSIM/BENICASIM',
    'BORRIANA/BURRIANA': 'BORRIANA/BURRIANA',
    'BURRIANA': 'BORRIANA/BURRIANA',
    'CABANAQUINTA/CABAÃ‘AQUINTA': 'ALLER', 
    'CASTELLON DE LA PLANA': 'CASTELLÓ DE LA PLANA/CASTELLÓN DE LA PLANA',
    'EJEA': 'EJEA DE LOS CABALLEROS',
    'EL GOLFO': 'YAIZA', 
    'ELCHE': 'ELX/ELCHE',
    'ELCHE/ELX': 'ELX/ELCHE',
    'ELEXALDE': 'GALDAKAO', 
    'JAVEA': 'XÀBIA/JÁVEA',
    'JAVEA/XABIA': 'XÀBIA/JÁVEA',
    'LA ALMUNIA': 'LA ALMUNIA DE DOÑA GODINA',
    'LA CONCHA': 'VILLAESCUSA', 
    'LA IGLESIA': 'RUILOBA', 
    'LA POLA': 'LENA', 
    'LAS PALMAS DE G.C.': 'LAS PALMAS DE GRAN CANARIA',
    'LLANSA': 'LLANÇÀ',
    'MATAMOROSA': 'CAMPOO DE ENMEDIO',
    'MIERES DEL CAMIN': 'MIERES',
    'MORON FRONTERA': 'MORÓN DE LA FRONTERA',
    'MURIEDAS': 'CAMARGO',
    'OSORNO': 'OSORNO LA MAYOR',
    'PALMA DE MALLORCA': 'PALMA', 
    'PAMPLONA': 'PAMPLONA/IRUÑA',
    'PEÃ‘ISCOLA': 'PENÍSCOLA/PEÑÍSCOLA',
    'POLLENÃ‡A': 'POLLENÇA',
    'POMALUENGO': 'CASTAÑEDA',
    'PUENTENANSA': 'RIONANSA',
    'RENEDO': 'PIÉLAGOS', 
    'RIVERO': 'SAN FELICES DE BUELNA',
    'RONCAL': 'RONCAL/ERRONKARI',
    'RUBAYO': 'MARINA DE CUDEYO',
    'SAN ILDEFONSO': 'REAL SITIO DE SAN ILDEFONSO',
    'SAN ILDEFONSO O LA GRANJA': 'REAL SITIO DE SAN ILDEFONSO',
    'SAN JOSE': 'NÍJAR', 
    'SAN MIGUEL DE LUENA': 'LUENA',
    'SAN MIGUEL DE MERUELO': 'MERUELO',
    'SAN VICENTE DE TORANZO': 'CORVERA DE TORANZO',
    'SANT CARLES DE LA RAPITA': 'LA RÀPITA', 
    'SETLA': 'ELS POBLETS', 
    'SIGÃœENZA': 'SIGÜENZA',
    'TAMA': 'CILLORIGO DE LIÉBANA', 
    'TARRIO': 'AMES', # Tarrio is ambiguous. It is a village not an official administrative municipio
    'TORRE BAJA': 'TORREBAJA',
    'TORRE ENDOMENECH': "LA TORRE D'EN DOMÉNEC",
    'UZTARROZ': 'UZTÁRROZ/UZTARROTZE',
    'VALDECILLA': 'MEDIO CUDEYO',
    'VEGUILLA': 'SOBA',
    'VILLAJOYOSA/LA VILA JOIOSA': 'LA VILA JOIOSA/VILLAJOYOSA',
    'VILLAYUSO': 'CIEZA'
}

In [24]:
final_corrections = matches | manual_corrections
final = pd.DataFrame.from_dict(final_corrections, orient='index').reset_index()
final.columns = ['DE_MUNICIP_org', 'DE_MUNICIP']
final

,DE_MUNICIP_org,DE_MUNICIP
0,A CORUÃ‘A,A CORUÑA
1,ABIA DE LAS TORRES,ABIA DE LAS TORRES
2,ABRERA,ABRERA
3,ACEBEDO,ACEBEDO
4,ACEUCHAL,ACEUCHAL
...,...,...
803,YUNQUERA,YUNQUERA
804,ZAFRA,ZAFRA
805,ZARAGOZA,ZARAGOZA
806,ZARZOSA DE RIO PISUERGA,ZARZOSA DE RÍO PISUERGA


In [25]:
final_merged = final.merge(latlong_data, on='DE_MUNICIP', how='left')
final_merged

,DE_MUNICIP_org,DE_MUNICIP,INE_CODE,PROVINCE_CODE,PROVINCE,LONGITUDE,LATITUDE,ALTITUDE
0,A CORUÃ‘A,A CORUÑA,1.503000e+10,15.0,A Coruña,-8.395826,43.371495,8
1,ABIA DE LAS TORRES,ABIA DE LAS TORRES,3.400300e+10,34.0,Palencia,-4.421902,42.420220,836
2,ABRERA,ABRERA,8.001000e+09,8.0,Barcelona,1.901569,41.516398,106
3,ACEBEDO,ACEBEDO,2.400100e+10,24.0,León,-5.115871,43.040732,1148
4,ACEUCHAL,ACEUCHAL,6.002000e+09,6.0,Badajoz,-6.487251,38.647755,303
...,...,...,...,...,...,...,...,...
812,YUNQUERA,YUNQUERA,2.910000e+10,29.0,Málaga,-4.917332,36.734203,695
813,ZAFRA,ZAFRA,6.158000e+09,6.0,Badajoz,-6.418559,38.426344,509
814,ZARAGOZA,ZARAGOZA,5.029700e+10,50.0,Zaragoza,-0.877318,41.656208,208
815,ZARZOSA DE RIO PISUERGA,ZARZOSA DE RÍO PISUERGA,9.482000e+09,9.0,Burgos,-4.267593,42.536190,826


In [26]:
final_merged.isna().sum()

DE_MUNICIP_org    0
DE_MUNICIP        0
INE_CODE          1
PROVINCE_CODE     1
PROVINCE          1
LONGITUDE         1
LATITUDE          1
ALTITUDE          1
dtype: int64

In [27]:
final_merged[final_merged['LATITUDE'].isna()]

,DE_MUNICIP_org,DE_MUNICIP,INE_CODE,PROVINCE_CODE,PROVINCE,LONGITUDE,LATITUDE,ALTITUDE
734,UZTARROZ,UZTÁRROZ/UZTARROTZE,NaN,NaN,NaN,NaN,NaN,NaN


In [28]:
all_districts = latlong_data['DE_MUNICIP'].tolist()
process.extract('UZTARROZ', all_districts, limit=5, scorer=fuzz.token_sort_ratio)

[('URROZ', 76.92307692307692, 7147),
 ('MAZARAMBROZ', 63.1578947368421, 4383),
 ('BULARROS', 62.5, 1340),
 ('ZARAGOZA', 62.5, 8061),
 ('BARRO', 61.53846153846154, 939)]

Distinct Municipalities: Uztárroz (Uztarrotze) and Urroz are two completely different, separate towns located in the province of Navarre.

In [29]:
final_merged[final_merged['DE_MUNICIP_org'].duplicated(keep=False)]

,DE_MUNICIP_org,DE_MUNICIP,INE_CODE,PROVINCE_CODE,PROVINCE,LONGITUDE,LATITUDE,ALTITUDE
131,CABANES,CABANES,1.203300e+10,12.0,Castelló/Castellón,0.045412,40.156104,290
132,CABANES,CABANES,1.703000e+10,17.0,Girona,2.977957,42.307504,24
199,CIEZA,CIEZA,3.001900e+10,30.0,Murcia,-1.427730,38.236594,187
200,CIEZA,CIEZA,3.902100e+10,39.0,Cantabria,-4.096722,43.221056,174
340,LA CONCHA,VILLAESCUSA,4.924100e+10,49.0,Zamora,-5.464099,41.206136,825
341,LA CONCHA,VILLAESCUSA,3.909900e+10,39.0,Cantabria,-3.856639,43.370333,35
434,MIERES,MIERES,1.710500e+10,17.0,Girona,2.640292,42.123296,282
435,MIERES,MIERES,3.303700e+10,33.0,Asturias,-5.772657,43.248794,210
436,MIERES DEL CAMIN,MIERES,1.710500e+10,17.0,Girona,2.640292,42.123296,282
437,MIERES DEL CAMIN,MIERES,3.303700e+10,33.0,Asturias,-5.772657,43.248794,210


In [30]:
# final_merged.to_csv(r'../BD_MUNICIPIOS-ENTIDADES/DE_MUNICIP_LAT_LONG.csv')

### HISTORECO: Historical Spanish transition database on climate, geography, and economics of the 20th-21st Century

https://figshare.com/articles/dataset/HISTORECO_Historical_Spanish_Transition_Database_on_Climate_Geography_and_Economics_of_the_20th-21st_Century/27262032/2?file=53497862

In [31]:
historeco = pd.read_csv(r'../BD_MUNICIPIOS-ENTIDADES/Historeco.csv', encoding='latin1')
historeco.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 105586 entries, 0 to 105585
Data columns (total 72 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   Unnamed: 0                  105586 non-null  int64  
 1   CODNUT2                     105586 non-null  object 
 2   ccaa                        105586 non-null  object 
 3   Province_code               105586 non-null  int64  
 4   Province                    105586 non-null  object 
 5   INEcode                     105586 non-null  int64  
 6   Municipality                105586 non-null  object 
 7   Year                        105586 non-null  object 
 8   pp                          105586 non-null  float64
 9   t_average                   105586 non-null  float64
 10  spei                        105586 non-null  float64
 11  grow_period_pp              105586 non-null  float64
 12  frost_days                  105586 non-null  float64
 13  dry_hot_climat

C:\Users\reema\AppData\Local\Temp\ipykernel_12056\2748759672.py:1: DtypeWarning: Columns (63) have mixed types. Specify dtype option on import or set low_memory=False.
  historeco = pd.read_csv(r'../BD_MUNICIPIOS-ENTIDADES/Historeco.csv', encoding='latin1')


In [32]:
historeco.Year.head()

0    1900s
1    1910s
2    1920s
3    1930s
4    1940s
Name: Year, dtype: object

In [33]:
historeco = historeco[historeco['Year'] == "2010s"]
historeco.shape

(8122, 72)

In [34]:
# Historeco variables description
var_des = pd.read_excel(r'../BD_MUNICIPIOS-ENTIDADES/Variable_description.xlsx')
var_des.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 71 entries, 0 to 70
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   Column       71 non-null     object
 1   Unit         71 non-null     object
 2   Description  71 non-null     object
 3   Class        71 non-null     object
dtypes: object(4)
memory usage: 2.3+ KB


In [35]:
var_des.Class.unique()

array(['Id', 'Climatic ', 'Geographical', 'Land use ', 'Hydrological ',
       'Socioeconomic-demographic '], dtype=object)

In [36]:
pd.set_option('display.max_colwidth', None)
var_des[var_des['Class'] == 'Geographical']

,Column,Unit,Description,Class
20,to_coast_km,Kilometers,Distance in kilometres from the centroid of the municipality to the coastline,Geographical
21,to_mad_km,Kilometers,Distance in kilometres from the centroid of the municipality to Madrid,Geographical
22,to_prov_cap_km,Kilometers,Distance in kilometres from the centroid of the municipality to the Province capital,Geographical
23,longitude,Degrees,Longitude of the centroid of the municipality,Geographical
24,latitude,Degrees,Latitude of the centroid of the municipality,Geographical
25,altitude,Meters,Average altitude above sea level of the municipality,Geographical
26,ruggedness,Meters,Standard deviation of the altitude of the municipality,Geographical
27,area,Square kilometers,Municipality area,Geographical


In [37]:
var_des[var_des['Class'] == 'Id']

,Column,Unit,Description,Class
0,CODNUT2,-,Identification code of the NUTS-2 region in which the municipality is located,Id
1,ccaa,-,Name of the autonomous community the municipality belongs to,Id
2,Province_code,-,INE Identification code of the province in which the municipality is located,Id
3,Province,-,Name of the province the municipality belongs to,Id
4,INEcode,-,INE municipality identification code,Id
5,Municipality,-,Municipality name,Id
6,Year,-,Decade to which the value belongs,Id


In [38]:
geographic_columns = var_des[var_des['Class'] == 'Geographical']["Column"].values
id_columns = var_des[var_des['Class'] == 'Id']["Column"].values
geo_historeco = historeco[list(id_columns)+list(geographic_columns)]
geo_historeco.head()

,CODNUT2,ccaa,Province_code,Province,INEcode,Municipality,Year,to_coast_km,to_mad_km,to_prov_cap_km,longitude,latitude,altitude,ruggedness,area
11,ES21,País Vasco,1,Alava,1001,Alegría-Dulantzi,2010s,52.680096,287.13525,13.061319,-2.53896,42.80243,636.186327,98.429316,19.824800
24,ES21,País Vasco,1,Alava,1002,Amurrio,2010s,30.246468,299.26764,23.517231,-2.99427,42.96977,409.196977,123.775589,96.123299
37,ES21,País Vasco,1,Alava,1003,Aramaio,2010s,31.984140,307.89203,23.784414,-2.59372,43.04610,627.271089,139.720063,73.214203
50,ES21,País Vasco,1,Alava,1004,Artziniega,2010s,24.392670,304.72101,22.613579,-3.14147,43.12971,314.590847,110.131148,27.377501
63,ES21,País Vasco,1,Alava,1006,Armiñón,2010s,68.034470,265.84329,21.724493,-2.89430,42.68872,505.879238,39.030491,10.591600


In [39]:
geo_historeco.isna().sum()

CODNUT2           0
ccaa              0
Province_code     0
Province          0
INEcode           0
Municipality      0
Year              0
to_coast_km       0
to_mad_km         0
to_prov_cap_km    0
longitude         0
latitude          0
altitude          1
ruggedness        1
area              0
dtype: int64

In [40]:
final_merged['INE5'] = (final_merged['INE_CODE'] // 1_000_000).astype('Int64')

# Historeco already stores INE5 (leading zeros dropped), so no change needed,
#    but rename for clarity:
geo_historeco = geo_historeco.rename(columns={'INEcode': 'INE5'})

print(final_merged['INE5'].head())
# 0     1503   → wait, check: 15030000000 // 1000000 = 15030 ✓

0    15030
1    34003
2     8001
3    24001
4     6002
Name: INE5, dtype: Int64


### Final data merge

In [41]:
merged_2 = final_merged[['DE_MUNICIP_org', 'INE5']].merge(geo_historeco, on='INE5', how='left')
merged_2.head()

,DE_MUNICIP_org,INE5,CODNUT2,ccaa,Province_code,Province,Municipality,Year,to_coast_km,to_mad_km,to_prov_cap_km,longitude,latitude,altitude,ruggedness,area
0,A CORUÃ‘A,15030,ES11,Galicia,15.0,a Coruña,A Coruña,2010s,0.186361,510.59659,0.177249,-8.45219,43.37695,75.107322,54.355599,37.987598
1,ABIA DE LAS TORRES,34003,ES41,Castilla y León,34.0,Palencia,Abia de las Torres,2010s,106.841140,231.52803,46.431713,-4.41905,42.43555,829.757739,18.765692,27.111401
2,ABRERA,8001,ES51,Cataluña,8.0,Barcelona,Abrera,2010s,28.033470,486.42380,27.359745,1.90998,41.51639,134.107796,49.198940,20.129200
3,ACEBEDO,24001,ES41,Castilla y León,24.0,León,Acebedo,2010s,46.559780,315.39801,61.069183,-5.12169,43.03373,1383.608978,193.161104,50.297699
4,ACEUCHAL,6002,ES43,Extremadura,6.0,Badajoz,Aceuchal,2010s,152.401540,310.11331,49.322891,-6.49078,38.66359,316.107055,27.376035,63.112400


In [42]:
merged_2.isna().sum()

DE_MUNICIP_org    0
INE5              1
CODNUT2           1
ccaa              1
Province_code     1
Province          1
Municipality      1
Year              1
to_coast_km       1
to_mad_km         1
to_prov_cap_km    1
longitude         1
latitude          1
altitude          2
ruggedness        2
area              1
dtype: int64

In [43]:
merged_2[merged_2['Municipality'].isna()]

,DE_MUNICIP_org,INE5,CODNUT2,ccaa,Province_code,Province,Municipality,Year,to_coast_km,to_mad_km,to_prov_cap_km,longitude,latitude,altitude,ruggedness,area
734,UZTARROZ,<NA>,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [44]:
missing_row = geo_historeco[
    (geo_historeco['INE5'] == 31247) &
    (geo_historeco['Year'] == '2010s')
]
missing_row

,CODNUT2,ccaa,Province_code,Province,INE5,Municipality,Year,to_coast_km,to_mad_km,to_prov_cap_km,longitude,latitude,altitude,ruggedness,area
63139,ES22,Navarra,31,Navarra,31247,Uztárroz/Uztarroze,2010s,80.446693,358.22754,58.067448,-0.95933,42.91482,1183.497252,185.587645,58.4688


In [45]:
for col in missing_row.columns:
    merged_2.loc[734, col] = missing_row[col].values[0]

merged_2.loc[734]

DE_MUNICIP_org              UZTARROZ
INE5                           31247
CODNUT2                         ES22
ccaa                         Navarra
Province_code                   31.0
Province                     Navarra
Municipality      Uztárroz/Uztarroze
Year                           2010s
to_coast_km                80.446693
to_mad_km                  358.22754
to_prov_cap_km             58.067448
longitude                   -0.95933
latitude                    42.91482
altitude                 1183.497252
ruggedness                185.587645
area                         58.4688
Name: 734, dtype: object

In [46]:
merged_2.isna().sum()

DE_MUNICIP_org    0
INE5              0
CODNUT2           0
ccaa              0
Province_code     0
Province          0
Municipality      0
Year              0
to_coast_km       0
to_mad_km         0
to_prov_cap_km    0
longitude         0
latitude          0
altitude          1
ruggedness        1
area              0
dtype: int64

In [47]:
# will deal with it later on 
merged_2[merged_2['altitude'].isna()]

,DE_MUNICIP_org,INE5,CODNUT2,ccaa,Province_code,Province,Municipality,Year,to_coast_km,to_mad_km,to_prov_cap_km,longitude,latitude,altitude,ruggedness,area
237,EL PINAR DE EL HIERRO,38901,ES70,Islas Canarias,38.0,Santa Cruz de Tenerife,El Pinar de El Hierro,2010s,1.349569,1935.496229,203.656953,-17.96451,27.68734,NaN,NaN,87.293394


### Final merged data export

In [48]:
# merged_2.to_csv(r'../BD_MUNICIPIOS-ENTIDADES/geo_encoded_munis.csv')

### Interactive Municipality Map

In [65]:
import plotly.express as px

fig = px.scatter_map(
    merged_2,
    lat="latitude",
    lon="longitude",
    hover_name="DE_MUNICIP_org",
    hover_data=["Province", "altitude", "to_coast_km", "area"],
    color="Province",
    zoom=4.5,
    height=700,
    map_style="carto-voyager", # or open-street-map
)
fig.update_traces(marker=dict(size=6, opacity=0.85))
fig.update_layout(margin=dict(l=0, r=0, t=0, b=0))
fig.show()

In [66]:
fig2 = px.scatter_map(
    merged_2,
    lat="latitude",
    lon="longitude",
    hover_name="DE_MUNICIP_org",
    color="altitude",
    color_continuous_scale=px.colors.sequential.Viridis,
    zoom=5,
    height=700,
    map_style="carto-darkmatter" # or "satellite-streets"
)

fig2.update_layout(margin=dict(l=0, r=0, t=0, b=0))
fig2.show()